<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/11_exgaussian_flat_priors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 11 — Flat population-level priors

Notebook 9 chose its priors so that each one said something plausible about reaction times. Notebook 10 replaced three of them with wide priors written directly on the log scale and never translated into milliseconds.

This notebook goes further and removes prior information altogether from the four population-level (or common-effect) parameters: the typical intercept and slope, the typical Gaussian standard deviation, and the tail mean shared by all participants. Their priors become flat, giving every value the same weight, while the priors for how much participants differ from one another stay Notebook 9's. Flat priors are often chosen in the hope of letting the data speak for themselves. Here they are chosen for you as a deliberate demonstration, not as a recommendation: the model is fitted to be diagnosed, not to be used. The purpose of this notebook is to see what a flat prior claims, which step of the workflow it removes, and whether the model can still be fitted.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The participant plotting helper from the previous notebooks is supplied. It can optionally restrict plots to selected participants using the standard ArviZ `coords` argument. Panels appear in participant order (308, 309, …, 372), left to right and top to bottom.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")
    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Removing prior information

### 1.1 The model

The model is Notebook 9's:

$$
y_i \sim \operatorname{ExGaussian}(\mu_{y,i}, sd_{y,s[i]}, \nu)
$$

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i
$$

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

$$
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1})
$$

$$
\log sd_{y,s}
\sim
\operatorname{Normal}(\mu_{\log sd_y}, sd_{\log sd_y}).
$$

Here $s[i]$ identifies the participant who produced observation $i$, and the tail mean $\nu$ is shared by all participants. Only the priors of the four population-level parameters change:

| Parameter | Notebook 9 | This notebook |
|---|---|---|
| `mu_b0`, $\mu_{b0}$ | $\operatorname{Normal}(250, 100)$ | flat |
| `mu_b1`, $\mu_{b1}$ | $\operatorname{Normal}(0, 20)$ | flat |
| `mu_log_sd_y`, $\mu_{\log sd_y}$ | $\operatorname{Normal}(\log 30, 0.5)$ | flat |
| `log_nu`, $\log \nu$ | $\operatorname{Normal}(\log 50, 0.75)$ | flat |

The priors for the three between-participant standard deviations, `sd_b0`, `sd_b1`, and `sd_log_sd_y`, are Notebook 9's, and so are the construction of `sd_y` and `nu` and the likelihood.

### 1.2 What does a flat prior claim?

PyMC writes a flat prior as `pm.Flat`, whose density has the same value for every real number. What does `pm.Flat("mu_b0")` say about the typical baseline location of the Gaussian part, in milliseconds? Can a density that has the same value everywhere be a probability distribution?

- answer here

### 1.3 What does a flat prior on $\log \nu$ say about the tail mean in milliseconds?

In Notebook 10 (Question 1.3), we translated a prior on $\log \nu$ into milliseconds. A flat prior on $\log \nu$ gives equal weight to intervals of equal width on the log scale. What does it say about tail means of 0.001–0.01 ms, 1–10 ms, and 10–100 ms? How much weight does it give to tails shorter than 1 ms, compared with Notebook 9's plausible range of 11–220 ms?

- answer here

### 1.4 The supplied model

The whole model is supplied in one cell. It is Notebook 9's code with `pm.Flat` for the four population-level parameters; Notebook 9's priors are shown in the comments. Notebook 9's `mean_rt` is left out, because no question here uses it.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the between-participant standard deviations, from Notebook 9
sd_sd_b0 = 25          # ms
sd_sd_b1 = 10          # ms/day
sd_sd_log_sd_y = 1 / 3

with pm.Model(coords=coords) as model:
    days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
    pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

    # Varying intercepts, with a flat population center
    mu_b0 = pm.Flat("mu_b0")  # Notebook 9: Normal(250, 100)
    sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
    b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

    # Varying slopes, with a flat population center
    mu_b1 = pm.Flat("mu_b1")  # Notebook 9: Normal(0, 20)
    sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
    b1 = pm.Normal("b1", mu=mu_b1, sigma=sd_b1, dims="participant")

    # Location of the Gaussian part (ms)
    mu_y = pm.Deterministic(
        "mu_y",
        b0[pidx] + b1[pidx] * days,
        dims="obs_id",
    )

    # Participants' Gaussian standard deviations, with a log link and a flat population center
    mu_log_sd_y = pm.Flat("mu_log_sd_y")  # Notebook 9: Normal(log 30, 0.5)
    sd_log_sd_y = pm.Exponential("sd_log_sd_y", scale=sd_sd_log_sd_y)
    log_sd_y = pm.Normal(
        "log_sd_y",
        mu=mu_log_sd_y,
        sigma=sd_log_sd_y,
        dims="participant",
    )
    sd_y = pm.Deterministic("sd_y", pm.math.exp(log_sd_y), dims="participant")

    # Shared tail mean, with a log link and a flat prior
    log_nu = pm.Flat("log_nu")  # Notebook 9: Normal(log 50, 0.75)
    nu = pm.Deterministic("nu", pm.math.exp(log_nu))

    y = pm.ExGaussian(
        "y",
        mu=mu_y,
        sigma=sd_y[pidx],
        nu=nu,
        observed=sleep["Reaction"].to_numpy(),
        dims="obs_id",
    )

### 1.5 Why does this notebook have no prior predictive check?

In Notebook 10, the prior predictive check exposed the priors' problems before any fitting (Question 2.6 there). What does a prior predictive check need to draw first, and why can it not do so here?

- answer here

## 2. Fit and diagnose the model

### 2.1 Sample from the posterior.

The model is sampled with `target_accept=0.97` and 2,500 tuning draws. These settings are not the cause of what follows: in test runs, PyMC's default settings gave even more divergences, and Notebook 9's `target_accept=0.99` left three of the four chains stuck. Expect sampling to take several minutes. When a sampler fails, the details of the failure can differ from one computer to another, and with this model even the overall picture can differ in one respect, explained in Question 2.5.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=2500,
        chains=4,
        target_accept=0.97,
        random_seed=RANDOM_SEED,
    )

### 2.2 Check population-level diagnostics.

Besides the divergences, the code reports how often the sampler reached its maximum tree depth (Notebook 10, Question 3.3), and its average step size. The summary and trace show `log_nu` rather than `nu`, because its draws span many orders of magnitude.

In [ ]:
sample_stats = idata["sample_stats"]
print("Divergences:", int(sample_stats["diverging"].sum().item()))
print(f"Draws at the maximum tree depth: {sample_stats['reached_max_treedepth'].mean().item():.1%}")
print(f"Average step size: {sample_stats['step_size'].mean().item():.4f}")

azs.summary(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y", "log_nu"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y", "log_nu"],
);

### 2.3 Screen all participants.

In [ ]:
participant_diagnostics = azs.summary(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    kind="diagnostics",
    round_to=2,
)

pd.DataFrame(
    {
        "value": [
            participant_diagnostics["r_hat"].max(),
            participant_diagnostics["ess_bulk"].min(),
            participant_diagnostics["ess_tail"].min(),
        ]
    },
    index=["largest R-hat", "smallest bulk ESS", "smallest tail ESS"],
)

### 2.4 Does the fit meet the diagnostic criteria?

Use the criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces. Read the warnings printed by the sampling cell as well as the summaries, and compare the scale of `log_nu` with Notebook 9's 90% HDI for $\nu$, about 3.7–11 ms, or about 1.3–2.4 on the log scale.

- answer here

### 2.5 Why can a chain sit far away from the others?

PyMC starts each chain with every flat parameter near zero: trajectories near 0 ms, Gaussian standard deviations of about 1 ms, and a tail mean of about 1 ms. From there, most chains move to trajectories like those of Notebook 9's fit. But a chain can instead settle where the Gaussian part has all but vanished: the Gaussian standard deviations shrink to thousandths of a millisecond or less, each participant's trajectory lies just below all of their observations, and a tail of tens to hundreds of milliseconds accounts for all of the scatter. Once the Gaussian part is that narrow, the data cannot tell it from no Gaussian part at all, and the flat prior on `mu_log_sd_y`, unlike Notebook 9's, does not rule out such tiny standard deviations. Every observation then acts as a wall (Notebook 9, Question 3.2), so the chain takes tiny steps, often reaches the maximum tree depth, and barely moves: it is stuck.

Whether a chain gets stuck depends on tiny numerical details, so it can differ from one computer to another, and even from one run to the next. In ten test runs of this notebook, about 40% of the chains got stuck, and nine of the ten runs had at least one stuck chain. In the one run in which no chain got stuck, the fit still had almost 2,000 divergences, but every population-level R-hat was 1.00. A stuck chain shows up as R-hat far above 1, as a trace that sits apart from the others, and in the chain means printed below.

In [ ]:
idata["posterior"].to_dataset()[["mu_b0", "mu_b1", "mu_log_sd_y", "log_nu"]].mean("draw").to_dataframe().round(2)

## 3. Read the failed fit

### 3.1 Where do the draws of `log_nu` lie?

A fit that fails its diagnostics is not reported, but its draws can still show what shaped them, and that helps explain the failure. Use the trace and density of `log_nu` in Question 2.2 and the chain means in Question 2.5. How are the draws of the chains that did not get stuck spread, and what tail means in milliseconds do the two ends of the spread correspond to?

- answer here

### 3.2 Why does nothing stop `log_nu` from decreasing?

In Notebook 10, the data bounded the tail only from above, and the $\operatorname{Normal}(0, 5)$ prior held the posterior back from ever shorter tails (Question 4.4 there). Once the tail is far shorter than every participant's Gaussian standard deviation, how does the likelihood change as $\log \nu$ decreases further? What does that imply for the posterior of $\log \nu$ under a flat prior?

- answer here

### 3.3 Then what stops the draws near −372, and where do the divergences come from?

The sampler's paths are steered by the slope of the log posterior density (Notebook 10, Question 3.3), and throughout the flat region that slope is exactly zero in the direction of `log_nu`, so nothing pulls the paths back. What stops them is the computer's arithmetic. A computer stores numbers in a limited range, and the smallest positive number it can store is about $5 \times 10^{-324}$. At $\log \nu \approx -372.5$, the tail mean is about $10^{-162}$ ms, and its square is too small to store: it becomes zero. Part of PyMC's slope calculation divides by that square, and once it is zero, the calculation produces *not a number* (NaN), a value that is no number at all. A path that reaches this point cannot continue, and the sampler reports a divergence. The lower edge of the draws is this limit of the arithmetic, not anything in the model or the data.

During tuning, the sampler learns how widely each parameter ranges and scales its steps for that parameter accordingly. For `log_nu`, the range is hundreds of units, so a single path can carry it a long way across the flat region, and many paths run into the limit. Here, divergences do not mark a part of the posterior that is too sharply curved for the sampler's steps (Notebook 9, Question 3.2); they mark the edge of what the computer can calculate. Increasing `target_accept`, as the warning suggests, makes the steps smaller, but no step size avoids that edge.

The failure differs from Notebook 10's. There, a proper prior kept $\log \nu$ within a range of about 20 units, where PyMC's switch to the Gaussian formula makes the density jump slightly; the sampler shrank its steps, almost every draw reached the maximum tree depth, and nothing diverged. Here, in the chains that reach the flat region, the steps are larger, almost no draw reaches the maximum tree depth, and about half of the draws diverge. The cause is the same: a prior that puts its weight where the data carry no information.

### 3.4 Summarize only the chains that reached the flat region.

Repeat the population-level summary of Question 2.2 for the chains that did not get stuck, selecting them with the `coords` argument, as for participants in Notebook 9 (Question 3.5). Use the chain means in Question 2.5 to decide which chains to keep.

In [ ]:
# answer here

### 3.5 What can R-hat and ESS detect here?

R-hat compares the chains with one another, and ESS estimates how many independent draws the chains are worth. Compare their values in your summary with those in Question 2.2. Does leaving out a stuck chain repair the fit?

- answer here

### 3.6 Generate and plot posterior predictive reaction times.

This fit's predictions will not be used, as in Notebook 10. The question here is different: had the diagnostics been skipped, would the posterior predictive check have raised an alarm? Generate replicated values of `y`, add them to `idata`, and plot them with `plot_participants`, as in Notebook 9 (Question 5.3).

In [ ]:
# answer here

### 3.7 Would this check have revealed the problem?

Compare the plot with Notebook 9's (Question 5.3 there). Would the predicted reaction times differ if the tail mean were $10^{-5}$ ms rather than $10^{-150}$ ms?

- answer here

### 3.8 Which results still look like Notebook 9's, and can they be reported?

Compare your summary in Question 3.4 with Notebook 9's: a 90% HDI for `mu_b1` of about 8.0–14.6 ms/day, `mu_b0` about 260 ms, and `mu_log_sd_y` about 2.8.

- answer here

## 4. Summary

### 4.1 What has this notebook shown?

Summarize what the flat priors claimed, which step of the workflow they removed, how sampling failed, and what the draws did and did not show.

- answer here